In [1]:
import pandas as pd
import numpy as np
from pandas import Series, DataFrame

In [2]:
before_df = (
    pd.read_csv('../../../data/nyc_taxi_2019-07.csv',
    header= 0, 
    sep= ',',
    usecols = ['passenger_count', 'total_amount', 'payment_type'])
)

after_df = (
    pd.read_csv('../../../data/nyc_taxi_2020-07.csv',
    header= 0, 
    sep= ',',
    usecols = ['passenger_count', 'total_amount', 'payment_type'])
)

In [3]:
before_df['year'] = 2019
after_df['year'] = 2020

In [4]:
df = pd.concat([before_df, after_df])
df

,passenger_count,payment_type,total_amount,year
0,1.0,1.0,4.94,2019
1,1.0,2.0,20.30,2019
2,1.0,1.0,70.67,2019
3,1.0,1.0,66.36,2019
4,0.0,1.0,15.30,2019
...,...,...,...,...
800407,NaN,NaN,83.50,2020
800408,NaN,NaN,19.78,2020
800409,NaN,NaN,38.45,2020
800410,NaN,NaN,29.77,2020


### How many rides were taken in 2019 and 2020, and what is the difference between these two figures?

In [5]:
print(df['year'].value_counts())
diff = ((df['year'] == 2019).sum()) - ((df['year'] == 2020).sum())
print()
print(f'This means in 2020 people took {diff:,} fewer taxi rides than in 2019')

year
2019    6310419
2020     800412
Name: count, dtype: int64

This means in 2020 people took 5,510,007 fewer taxi rides than in 2019


### How much mmorey (in total) was collected in 2019 and 2020, and what was the difference between these two figures?

In [6]:
total_amount_2019 = df.loc[df['year'] == 2019, 'total_amount'].sum()
total_amount_2020 = df.loc[df['year'] == 2020, 'total_amount'].sum()

total_amount_diff = total_amount_2019 - total_amount_2020

print(f'{total_amount_diff:,.0f}')


108,848,979


### Did the proportion of trips with more than more passenger change dramatically?

In [7]:
more_passanger__ratio_2019 = len(df.query('year == 2019 & passenger_count > 1')) / len(df.query('year == 2019'))

In [8]:
more_passanger__ratio_2019

0.2818649601555776

In [9]:
more_passanger__ratio_2020 = len(df.query('year == 2020 & passenger_count > 1')) / len(df.query('year == 2020'))

In [10]:
more_passanger__ratio_2020

0.18996466819587912

##### This two cell below is a note for myself

In [11]:
more_passanger_ratio_2019 = (
    df.query('year == 2019 & passenger_count > 1')['passenger_count'].count() /
    df.query('year == 2019')['payment_type'].count() # NOTE:Reuven used count() to exclude Nan values
)

print(f'{more_passanger_ratio_2019:,.4f}')

0.2834


In [12]:
more_passanger_ratio_2020 = (
    df.query('year == 2020 & passenger_count > 1')['passenger_count'].count() /
    df.query('year == 2020')['payment_type'].count() # NOTE:Reuven used count() to exclude Nan values
)

print(f'{more_passanger_ratio_2020:,.4f}')

0.2062


### Did people use cash (i.e., payment_type of 2) less in 2020 than in 2019?

In [13]:
cash_ratio_2019 = (
    df.query('year == 2019 & payment_type == 2')['payment_type'].count() /
    df.query('year == 2019')['payment_type'].count() # NOTE:Reuven used count() to exclude Nan values
)

print(f'{cash_ratio_2019:,.4f}')

0.2871


In [14]:
cash_ratio_2020 = (
    df.query('year == 2020 & payment_type == 2')['payment_type'].count() /
    df.query('year == 2020')['payment_type'].count() # NOTE:Reuven used count() to exclude Nan values
)

print(f'{cash_ratio_2020:,.4f}')

# Interestingly, the proportion of cash payments increased by about ~3%, even though I wouldn’t have expected that.


0.3206


### Use the corr `method` on `df` to find the correlations among the columns. How would you interpret these results?

In [15]:
df.corr()

# No significant correlation is observed.

,passenger_count,payment_type,total_amount,year
passenger_count,1.000000,0.016410,0.014943,-0.049558
payment_type,0.016410,1.000000,-0.138561,0.029277
total_amount,0.014943,-0.138561,1.000000,-0.019706
year,-0.049558,0.029277,-0.019706,1.000000


### Show, with a single command, the difference in descriptive statistics for `total_amount` between 2019 and 2020. Round values to use no more than two digits after the decimal point

In [16]:
(
    df.loc[df["year"] == 2020, "total_amount"].describe()
    - df.loc[df["year"] == 2019, "total_amount"].describe()
).round(2)


count   -5510007.00
mean          -0.98
std           -0.75
min           53.20
25%           -0.50
50%           -0.60
75%           -0.75
max        -4672.45
Name: total_amount, dtype: float64

### If we assume that zero-passenger trips are for delivering packages, how were those affected during the pandemic? Show the proportion of such trips in 2019 versus 2020.

In [17]:
(
    df.loc[(df["passenger_count"] == 0) & (df["year"] == 2019), "year"].count() /
    df.loc[(df["year"] == 2019), "year"].count()
).round(4)

np.float64(0.0185)

In [20]:
(
    df.loc[(df["passenger_count"] == 0) & (df["year"] == 2020), "year"].count() /
    df.loc[(df["year"] == 2020), "year"].count()
).round(4)

# The proportion of zero-passenger trips increased during the pandemic.

np.float64(0.0244)